In [268]:
import pandas as pd
from pandas import DataFrame
import numpy as np
import networkx as nx
import peartree as pear
import partridge as ptg
import matplotlib.pyplot as plt
import itertools
import warnings
import json
warnings.filterwarnings('ignore')

In [269]:
feed_all = pear.get_representative_feed("gtfs_subway/")
feed = feed_all

In [270]:
def iterator(L):
	a, b = itertools.tee(L)
	next(b, None)
	return list(zip(a,b))

In [271]:
def enum(x):
	i = range(len(x))
	l = [list(k) for k in zip(x, i)]
	return l

def enumone(x):
	i = range(1, len(x)+1)
	l = [list(k) for k in zip(x, i)]
	return l

def intersect(x,y):
	return(set(x) & set(y))

def set_diff(x,y): return(set(x) - set(y))

In [272]:
station_data = pd.read_csv("data/MTA_Subway_Stations.csv")
station_data = station_data[['GTFS Stop ID', 'Complex ID', 'Stop Name', 'Daytime Routes']]

rl = [str(r).split() for r in station_data['Daytime Routes']]
station_data['Daytime Routes'] = rl

# remove Staten Island Railroad

station_data = station_data[station_data['Daytime Routes'].apply(lambda x: "SIR" not in x)]
station_data.set_index('GTFS Stop ID', drop=False, inplace=True)
nodes_list = [(k, v) for k,v in station_data.to_dict('index').items()]

In [273]:
line_names = [str(s) for s in [1,2,3,4,5,6,'6X',7,'7X','A','B','C','D','E','F','FX','G','H','J','L','M','N','Q','R','W','Z','FS','GS','Transfer']]

In [274]:
line_order = sorted(line_names)
line_order

['1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '6X',
 '7',
 '7X',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'FS',
 'FX',
 'G',
 'GS',
 'H',
 'J',
 'L',
 'M',
 'N',
 'Q',
 'R',
 'Transfer',
 'W',
 'Z']

In [275]:
## transfers in complex
transfers = pd.read_csv('gtfs_subway/transfers.txt')
	
transfers.drop("transfer_type", inplace=True, axis=1)

transfers = transfers[transfers['from_stop_id'] != transfers['to_stop_id']]
transfers.reset_index(inplace=True,drop=True)
transfers

,from_stop_id,to_stop_id,min_transfer_time
0,112,A09,180
1,125,A24,180
2,127,725,180
3,127,902,180
4,127,A27,300
...,...,...,...
147,R31,235,180
148,R31,D24,300
149,R33,F23,180
150,S01,A45,180


In [276]:
complexes = pd.read_csv("data/MTA_Subway_Stations_and_Complexes_20250918.csv")
complexes = complexes[complexes['Number Of Stations In Complex'] > 1]
complexes = complexes[['Complex ID','Number Of Stations In Complex', 'GTFS Stop IDs']]
gtfs_ids = [ids.split("; ") for ids in complexes['GTFS Stop IDs']]
complex_pairs = [list(itertools.combinations(sts, 2)) for sts in gtfs_ids]
complexes['complex_transfers'] = complex_pairs
complexes = complexes.explode('complex_transfers', ignore_index=True).drop(['Number Of Stations In Complex','GTFS Stop IDs'], axis= 1)
transfer_edges = list(zip(transfers['from_stop_id'], transfers['to_stop_id']))
transfer_edge_set = set()
for k in transfer_edges:
	transfer_edge_set.add(tuple(sorted(k)))

##print(transfer_edge_set ^ set(complexes['complex_transfers']))
##('254', 'L26'), ('629', 'B08'), ('B08', 'R11')##

## Out of station transfers specified on the official map
## Walk Junius to Livonia
## Walk Lexington-59 NRW to Lexington-63 -- 300s
## walk Lexington-59 456 to Lexington-63 -- 300s
transfers['edges'] = [tuple(sorted(k)) for k in transfer_edges]
unique_transfers = transfers.sort_values('edges').iloc[::2,:]
unique_transfers.reset_index(drop=True,inplace=True)
unique_transfers.drop(['from_stop_id', 'to_stop_id'], axis=1, inplace=True)

In [277]:
unique_transfers.edges.to_csv('transfer_edges.csv')

In [278]:
fstop = [unique_transfers['edges'][k][0] for k in unique_transfers.index]
tstop = [unique_transfers['edges'][k][1] for k in unique_transfers.index]
ttime = unique_transfers['min_transfer_time']
transfer_edge_weights = {(a, b): {"Transfer": t} for a,b,t in list(zip(fstop, tstop, ttime))}

### Fix edge weights so that boarding cost varies by line

In [279]:
start_time_am = int(6.5 * 60 * 60)
end_time_am = int(9.5 * 60 * 60)

In [280]:
start_time_pm = int(15.5 * 60 * 60)
end_time_pm = int(20 * 60 * 60)
tripline = list(feed.stop_times['trip_id'])
inseq = feed.stop_times.copy()
tripdict = {k: v for k,v in zip(feed.trips['trip_id'], feed.trips['route_id'])}
feed.stop_times['lines'] = [tripdict[s] for s in feed.stop_times['trip_id']]
feed.stop_times['dir'] = [s[-4] if s[-4] != 'X' else s[-7] for s in feed.stop_times['trip_id']]
stoptime_a = pd.DataFrame(feed.stop_times).copy()
stoptime_morning = stoptime_a[stoptime_a['arrival_time'].between(start_time_am, end_time_am, inclusive='both')]
stoptime_evening = stoptime_a[stoptime_a['arrival_time'].between(start_time_pm, end_time_pm, inclusive='both')]

In [281]:
def graphify(stoptime: DataFrame, debug=False):
	stoptime = stoptime[stoptime['lines'] != 'SI']
	stoptime.sort_values(['stop_id','lines', 'dir', 'arrival_time'], inplace=True)
	seq_dict = {}
	for n, g in inseq.groupby('trip_id'):
		seq_dict[n] = len(g['stop_sequence'])
	stoptime.sort_values('trip_id', inplace=True)
	stoptime['trip_len'] = [seq_dict[n] for n in stoptime['trip_id']]
	stops_in_time = dict(stoptime.trip_id.value_counts())
	stoptime['count_in_timespan'] = [stops_in_time[n] for n in stoptime.trip_id]
	stoptime = stoptime[stoptime['count_in_timespan'] == stoptime['trip_len']]
	stgroups = stoptime.groupby(['stop_id', 'lines','dir'], group_keys=True)
	st_weights = pd.DataFrame(list(stgroups.groups.keys()),index=list(stgroups.groups.keys()), columns=['stop', 'line', 'dir']).reset_index(drop=False)
	st_weights['weight'] = 0
	i = 0
	for name, group in stgroups:
		waittime = iterator(list(group['arrival_time']))
		dirw = round(np.mean([b-a for a,b in waittime]), 3)
		st_weights.at[i, 'weight'] = dirw
		i += 1


	st_weights = st_weights.groupby(['stop','line'], as_index=False).mean('weight')
	st_weights.sort_values('stop', inplace=True)

	node_weight_dict = {}
	for name, group in st_weights.groupby(['stop'],group_keys=True):
		float_w = dict(zip(group['line'], group['weight']))
		float_w = {k: round(v,3) for k,v in float_w.items()}
		node_weight_dict[name[0]] = float_w
	
	stops_per_line = stoptime.sort_values(['lines','trip_len'])
	stops_per_line = stops_per_line.groupby('trip_id', group_keys=False).apply(lambda x: x.sort_values('stop_sequence'))
	edge_set = set()
	for name, group in stops_per_line.groupby('trip_id'):
		edges = iterator(group.stop_id.to_list())
		for k in edges:
			edge_set.add(tuple(sorted(k)))
	
	w = {e: {} for e in edge_set}
	direction = {e: '' for e in edge_set}
	stoptime = stoptime.groupby('trip_id', as_index=False, group_keys=False).apply(lambda x: x.sort_values('stop_sequence', ascending=True))
	for name, group in stoptime.groupby('trip_id'):
		line = group.lines.values[0]
		delta = group['arrival_time'].diff()
		stop = list(zip(list(zip(group['stop_id'], group['stop_id'].shift())), delta))[1:]
		for e,t in stop:
			e = tuple(sorted(list(e)))
			try:
				w[e][line]
			except KeyError:
				w[e][line] = []
			finally:
				w[e][line].append(t)

	def roundlist(x):
		if type(x) == list: return int(np.ceil(np.mean(x)))
		else: return x

	all_edge_times = pd.DataFrame.from_dict(w).T
	avg_times = all_edge_times.copy()
	meandf = avg_times.map(lambda x: roundlist(x))
	edge_weights = meandf.apply(lambda x: x.dropna().to_dict(),axis=1).to_dict()
	edge_weights = edge_weights | transfer_edge_weights
	exp = pd.DataFrame(edge_weights.items(), columns=['stops', 'time'])
	exp.set_index("stops", inplace=True)
	exp['time'] = [list(k.items()) for k in exp['time']]
	exp = exp.explode('time').reset_index()
	exp['line'] = [t[0] for t in exp.time]
	exp['time'] = [t[1] for t in exp.time]
	exp = exp[['stops', 'line', 'time']]
	exp['order'] = [line_order.index(x) for x in exp['line']]
	exp = exp.sort_values('order').drop('order', axis=1)
	exp['edge_direction'] = [direction.get(w, 'T') for w in exp['stops']]	
	zipkey = zip(exp.stops, exp.line)
	keyed = [(i[0],i[1],j) for i,j in zipkey]
	edge_dict = dict({k: v for k,v in zip(keyed, exp.time)})
	return node_weight_dict, edge_dict

In [282]:
am_nodes, am_edges = graphify(stoptime_morning)
pm_nodes, pm_edges = graphify(stoptime_evening)

In [283]:
def get_nans(nodes: dict):
	nanlist = []
	for k,v in nodes.items():
		for k2,v2 in v.items():
			if np.isnan(v2):
				nanlist.append([k,k2])

	return nanlist

In [284]:
for k1,k2 in get_nans(pm_nodes):
	try:
		del pm_nodes[k1][k2]
	except KeyError:
		continue

In [285]:
for k1,k2 in get_nans(am_nodes):
	try:
		del am_nodes[k1][k2]
	except KeyError:
		continue

In [286]:
ndfam, ndfpm = pd.DataFrame.from_dict(am_nodes).T,pd.DataFrame.from_dict(pm_nodes).T

In [287]:
ndfpm

,1,2,3,5,4,6,6X,7,7X,GS,...,N,Q,M,FS,J,Z,R,H,L,W
101,268.157,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
103,251.843,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
104,251.536,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
106,251.536,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
107,251.536,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
R44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,428.15,NaN,NaN,NaN
R45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,428.15,NaN,NaN,NaN
S01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,600.0,NaN,NaN,NaN,NaN,NaN,NaN
S03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,600.0,NaN,NaN,NaN,NaN,NaN,NaN


In [288]:
ndnp = np.nanmean(np.stack([ndfam.to_numpy(), ndfpm.to_numpy()]),axis=0)
ndfr = pd.DataFrame(ndnp, index = pd.DataFrame.from_dict(am_nodes).T.index.values, columns=ndfam.columns)

In [289]:
ndfr

,1,2,3,5,4,6X,6,7,7X,GS,...,N,Q,M,FS,J,Z,R,H,L,W
101,334.9365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
103,295.8560,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
104,295.7025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
106,295.7025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
107,295.7025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
R44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,435.666,NaN,NaN,NaN
R45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,434.984,NaN,NaN,NaN
S01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,601.3235,NaN,NaN,NaN,NaN,NaN,NaN
S03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,601.3235,NaN,NaN,NaN,NaN,NaN,NaN


In [290]:
nodedict = ndfr.to_dict('index')
for k1,k2 in get_nans(nodedict):
	try:
		del nodedict[k1][k2]
	except KeyError:
		continue

In [291]:
edfpm = pd.DataFrame.from_dict(pm_edges, 'index', columns=['time']).reset_index(names='edge')
edfam = pd.DataFrame.from_dict(am_edges, 'index', columns=['time']).reset_index(names='edge')
edf = pd.concat([edfam, edfpm])
edf['line'] = edf.edge.map(lambda w: w[2])
edf['parent_edge'] = edf.edge.map(lambda w: w[:2])

In [292]:
edf = edf.sort_values('edge').groupby('edge').agg({'parent_edge':'first', 'line':'first','time':'mean'})

In [293]:
expdf = edf.reset_index()
expdf = expdf[expdf.line != 'Transfer']
expdf['stop1'] = expdf.edge.map(lambda w: (w[0],w[2]))
expdf['stop2'] = expdf.edge.map(lambda w: (w[1],w[2]))
edge_key = expdf.copy()[['stop1','stop2','time']]

In [294]:
edge_transfer =  edf.reset_index()
edge_transfer = edge_transfer[edge_transfer.line == 'Transfer']
edge_transfer['stop1'] = edge_transfer.edge.map(lambda w: w[0])
edge_transfer['stop2'] = edge_transfer.edge.map(lambda w: w[1])
edge_transfer.drop(['edge','parent_edge','line'],axis=1,inplace=True)
edge_transfer

,time,stop1,stop2
10,180.0,112,A09
30,180.0,125,A24
35,180.0,127,725
36,180.0,127,902
37,300.0,127,A27
...,...,...,...
793,180.0,L03,R20
805,180.0,L17,M08
841,180.0,M20,Q01
842,300.0,M20,R23


In [295]:
nodedict

{'101': {'1': 334.9365},
 '103': {'1': 295.856},
 '104': {'1': 295.7025},
 '106': {'1': 295.7025},
 '107': {'1': 295.7025},
 '108': {'1': 295.7025},
 '109': {'1': 295.7025},
 '110': {'1': 295.7025},
 '111': {'1': 295.7025},
 '112': {'1': 295.7025},
 '113': {'1': 295.9805},
 '114': {'1': 295.9805},
 '115': {'1': 268.9465},
 '116': {'1': 268.5895},
 '117': {'1': 269.1965},
 '118': {'1': 269.1965},
 '119': {'1': 268.8395},
 '120': {'1': 268.8395, '2': 403.82349999999997, '3': 452.91650000000004},
 '121': {'1': 268.48199999999997},
 '122': {'1': 268.993},
 '123': {'1': 268.48199999999997, '2': 404.3595, '3': 453.3855},
 '124': {'1': 268.6355},
 '125': {'1': 268.6355},
 '126': {'1': 268.6355},
 '127': {'1': 269.0385, '2': 405.142, '3': 454.01},
 '128': {'1': 269.0385, '2': 405.142, '3': 454.01},
 '129': {'1': 268.9315},
 '130': {'1': 268.9315},
 '131': {'1': 268.9315},
 '132': {'1': 268.9315, '2': 404.3085, '3': 453.385},
 '133': {'1': 268.9315},
 '134': {'1': 268.9315},
 '135': {'1': 269.0

In [296]:
node_parts = pd.DataFrame.from_dict({k: [list(v.keys())] for k,v in nodedict.items()}, 'index')
node_parts.reset_index(inplace=True)
node_parts.columns = ['stop','lines']
node_parts['same_stop'] = np.arange(len(node_parts))
nodes=node_parts.copy()
node_parts = node_parts.explode('lines')

In [297]:
node_parts['time'] = [nodedict[a][b] for a,b in zip(node_parts.stop, node_parts.lines)]

In [298]:
transfers = pd.concat([edge_transfer['stop1'],edge_transfer['stop2']]).unique()
node_transfer_edges = nodes[[k in transfers for k in nodes['stop']]].reset_index()
line_dict = dict(zip(node_transfer_edges['stop'], node_transfer_edges['lines']))
edge_transfer['key1'] = [line_dict[n] for n in edge_transfer['stop1']]
edge_transfer['key2'] = [line_dict[n] for n in edge_transfer['stop2']]
edge_transfer = edge_transfer[['stop1', 'stop2', 'key1', 'key2','time']]
node_transfer_edges['same_stop'] = np.arange(len(node_transfer_edges))
stop_dict = dict(zip(node_transfer_edges['stop'], node_transfer_edges['same_stop']))
te1 = edge_transfer.copy()
edge_transfer = edge_transfer.explode('key1').explode('key2')

In [299]:
gn = nx.DiGraph(name='dimensional')
gn.add_weighted_edges_from(zip(edge_key['stop1'], edge_key['stop2'], edge_key['time']), weight='travel_time', transfer=0,same_stop=0)
gn.add_weighted_edges_from(zip(edge_key['stop2'], edge_key['stop1'], edge_key['time']), weight='travel_time', transfer=0,same_stop=0)

In [300]:
te1 = te1.explode('key1').explode('key2')

In [301]:
te = zip(zip(te1['stop1'], te1['key1']), zip(te1['stop2'], te1['key2']), te1['time'])
te = pd.DataFrame(te, columns=['stop1','stop2','time'])
te2 = te[['stop2','stop1','time']]
te = pd.concat([te,te2]).reset_index(drop=True)
te['transfer'] = 1
te['in_complex'] = 0
te.rename(columns={'time':'travel_time'}, inplace=True)
te

,stop1,stop2,travel_time,transfer,in_complex
0,"(112, 1)","(A09, A)",180.0,1,0
1,"(112, 1)","(A09, C)",180.0,1,0
2,"(125, 1)","(A24, A)",180.0,1,0
3,"(125, 1)","(A24, C)",180.0,1,0
4,"(125, 1)","(A24, B)",180.0,1,0
...,...,...,...,...,...
789,"(M20, Z)","(R23, W)",300.0,1,0
790,"(Q01, N)","(R23, R)",180.0,1,0
791,"(Q01, N)","(R23, W)",180.0,1,0
792,"(Q01, Q)","(R23, R)",180.0,1,0


In [302]:
gn.add_weighted_edges_from(zip(te['stop1'], te['stop2'], te['travel_time']), weight='travel_time', transfer=1,same_stop=0)
gn.add_weighted_edges_from(zip(te['stop2'], te['stop1'], te['travel_time']), weight='travel_time', transfer=1,same_stop=0)

In [303]:
node_parts['node'] = list(zip(node_parts.stop, node_parts.lines))
node_parts.set_index('node',inplace=True)

In [304]:
for name, group in node_parts.groupby('same_stop'):
	if len(group) == 1: continue
	else:
		for u,v in list(itertools.permutations(group.index.to_list(),2)):
			gn.add_weighted_edges_from([(u,v,group['time'][v])], weight='travel_time', transfer=1, same_stop=1)
			gn.add_weighted_edges_from([(v,u,group['time'][u])], weight='travel_time', transfer=1, same_stop=1)

In [305]:
edgedict = edf['time'].to_dict()

In [306]:
f = open("subwaygraph_midday_dir.json",'r')
gd = nx.node_link_graph(json.load(f), directed=False, multigraph=True, edges='edges')

In [307]:
gn.edges.data()

OutEdgeDataView([(('101', '1'), ('103', '1'), {'transfer': 0, 'same_stop': 0, 'travel_time': 119.0}), (('103', '1'), ('104', '1'), {'transfer': 0, 'same_stop': 0, 'travel_time': 94.0}), (('103', '1'), ('101', '1'), {'transfer': 0, 'same_stop': 0, 'travel_time': 119.0}), (('104', '1'), ('106', '1'), {'transfer': 0, 'same_stop': 0, 'travel_time': 90.0}), (('104', '1'), ('103', '1'), {'transfer': 0, 'same_stop': 0, 'travel_time': 94.0}), (('106', '1'), ('107', '1'), {'transfer': 0, 'same_stop': 0, 'travel_time': 90.0}), (('106', '1'), ('104', '1'), {'transfer': 0, 'same_stop': 0, 'travel_time': 90.0}), (('107', '1'), ('108', '1'), {'transfer': 0, 'same_stop': 0, 'travel_time': 73.5}), (('107', '1'), ('106', '1'), {'transfer': 0, 'same_stop': 0, 'travel_time': 90.0}), (('108', '1'), ('109', '1'), {'transfer': 0, 'same_stop': 0, 'travel_time': 90.0}), (('108', '1'), ('107', '1'), {'transfer': 0, 'same_stop': 0, 'travel_time': 73.5}), (('109', '1'), ('110', '1'), {'transfer': 0, 'same_stop':

In [309]:
colors = ['#0062CF','#EB6800','#799534','#8E5C33','#7C858C','#F6BC26','#7C858C','#D82233','#009952','#9A38A1']
lines_color = [n.split(",") for n in ["A,C,E","B,D,F,FX,M","G","J,Z","L","N,Q,R,W","FS,GS,H","1,2,3","4,5,6,6X","7,7X",]]
colorcode = pd.DataFrame([lines_color,colors]).T
colorcode.columns=['line', 'color']
colorcode = colorcode.explode('line').set_index('line')
cold = {k: list(v.values())[0] for k,v in colorcode.to_dict('index').items()}

In [326]:
ndata = pd.DataFrame.from_dict(nodedict.items())
ndata.columns=['stop','wait_time']
ndata['lines'] = ndata.wait_time.map(lambda d: list(d.keys()))
ndata.wait_time = ndata.wait_time.map(lambda d: list(d.values()))
ndata['same_stop'] = np.arange(len(ndata))
ndata = ndata.explode(['lines', 'wait_time'])
ndata.reset_index(drop=True, inplace=True)
ndata.sort_values(['same_stop','lines'], inplace=True)
xy = pd.read_csv('gtfs_subway/stops.txt')
xy = xy[[len(k) == 3 for k in xy['stop_id']]]
xy = xy.rename({'stop_lat': 'y', 'stop_lon':'x'}, axis=1)[['stop_id','x','y']].set_index('stop_id')
xyd = xy.to_dict('index')
ndata['x'] = [xyd[n]['x'] for n in ndata['stop']]
ndata['y'] = [xyd[n]['y'] for n in ndata['stop']]
ndata['color'] = [cold[s] for s in ndata['lines']]

In [331]:
ndata.index = list(zip(ndata.stop,ndata.lines))

In [334]:
nx.set_node_attributes(gn, ndata.to_dict('index'))

In [335]:
gn.nodes.data()

NodeDataView({('101', '1'): {'stop': '101', 'wait_time': 334.9365, 'lines': '1', 'same_stop': 0, 'x': -73.898583, 'y': 40.889248, 'color': '#D82233'}, ('103', '1'): {'stop': '103', 'wait_time': 295.856, 'lines': '1', 'same_stop': 1, 'x': -73.90087, 'y': 40.884667, 'color': '#D82233'}, ('104', '1'): {'stop': '104', 'wait_time': 295.7025, 'lines': '1', 'same_stop': 2, 'x': -73.904834, 'y': 40.878856, 'color': '#D82233'}, ('106', '1'): {'stop': '106', 'wait_time': 295.7025, 'lines': '1', 'same_stop': 3, 'x': -73.909831, 'y': 40.874561, 'color': '#D82233'}, ('107', '1'): {'stop': '107', 'wait_time': 295.7025, 'lines': '1', 'same_stop': 4, 'x': -73.915279, 'y': 40.869444, 'color': '#D82233'}, ('108', '1'): {'stop': '108', 'wait_time': 295.7025, 'lines': '1', 'same_stop': 5, 'x': -73.918822, 'y': 40.864621, 'color': '#D82233'}, ('109', '1'): {'stop': '109', 'wait_time': 295.7025, 'lines': '1', 'same_stop': 6, 'x': -73.925536, 'y': 40.860531, 'color': '#D82233'}, ('110', '1'): {'stop': '110',

## export

In [336]:
def serialize(obj):
    if isinstance(obj, np.int64):
         return int(obj)
    if not isinstance(obj, (str, int, float, bool)):
        return list(obj)
    else: return obj

data1 = nx.node_link_data(gn, edges="edges")
with open('subwaygraph_rush_dir.json', 'w') as f:
    json.dump(data1,f, default=serialize)